# Pipelines

Pipelines are systems which automate the entire ML training process, driven purely from a declarative YAML configuration.

In [ ]:
import os
import flatiron.api as fi

<font color='#93B6E6'>A YAML configuration for a dummy PyTorch model.

In [39]:
uri = os.environ.get('SLACK_URI', '')
config = f'''
framework:
    name: torch
    device: cuda
model:
    input_channels: 3
    output_channels: 1
dataset:
    source: /mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001
    ext_regex: exr
    labels: ['a']
    label_axis: -1
    limit: 100
    test_size: 0.2
    reshape: False
optimizer:
    name: SGD
loss:
    name: MSELoss
metrics: []
callbacks:
    project: unet001
    root: /mnt/storage/projects
train:
    batch_size: 16
logger:
    slack_url: 'https://hooks.slack.com/services/{uri}'
    slack_channel: dev
    slack_methods:
        - train
    timezone: 'America/Detroit'
'''

<font color='#93B6E6'>DummyPipeline (a custom subclass of PipelineBase) reads its configuration
as a YAML string.

In [40]:
pipe = fi.torch.models.dummy.DummyPipeline.from_string(config)
pipe

<font color='#93B6E6'>To run the pipeline just call **run**. 

<font color='#93B6E6'>This will do everything from load your data, to create a tensorboard project, to train your model and send you Slack messages for each step.

Note: Logger is set to warn by default.

In [41]:
pipe.run()

  0%|          | 0/30 [00:00<?, ?it/s]


RUN TIME:
```26.25 seconds (0:00:26.250205)```
POST TIME:
```2025-03-21T15:03:08.698225-04:00```
CONFIG:
```callbacks:
    initial_value_threshold: null
    mode: auto
    monitor: val_loss
    project: unet001
    root: /mnt/storage/projects
    save_best_only: false
    save_freq: epoch
    save_weights_only: false
    verbose: 0
dataset:
    ext_regex: exr
    label_axis: -1
    labels:
    - a
    limit: 100
    reshape: false
    seed: null
    shuffle: true
    source: /mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001
    test_size: 0.2
framework:
    device: cuda
    name: torch
logger:
    level: warn
    slack_channel: dev
    slack_methods:
    - train
    slack_url: https://hooks.slack.com/services/
    timezone: America/Detroit
loss:
    name: MSELoss
    reduce: null
    reduction: mean
    size_average: null
metrics: []
model:
    input_channels: 3
    output_channels: 1
optimizer:
    dampening: 0
    differentiable: false
    foreach: null
    fused

<font color='#93B6E6'>Pipelines are design with a chainable builder pattern. Calling **run** is the same as calling the following:

In [ ]:
pipe \
    .build() \
    .compile() \
    .train_test_split() \
    .train()

<font color='#93B6E6'>For Tensorflow pipelines, **run** is the same as the following:

In [ ]:
pipe \
    .build() \
    .compile() \
    .train_test_split() \
    .load() \
    .train()

### Design Principle

<font color='#93B6E6'>The design principle here is simple:

<font color='#EBB483'>Validate every last argument to a pipeline before ever calling a single line of business logic.

<font color='#93B6E6'>Pipelines resolve fields in their configs to custom Pydantic configs which represent the call signature
<font color='#93B6E6'>of an underlying class or function within a given framework. Thes configs are then populated with default values and validated.
<font color='#93B6E6'>Validating configs as early as possible prevents framework code from every being called with invalid arguments.

<font color='#93B6E6'>In the later stages of the pipeline, actual framework classes or functions are then called with these config values.
<font color='#93B6E6'>Supported framework include: PyTorch and Tensorflow.

<font color='#93B6E6'>The final form of this config is shown below:

In [6]:
pipe.config

{'framework': {'name': 'torch', 'device': 'cuda'},
 'dataset': {'source': '/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
  'ext_regex': 'exr',
  'labels': ['a'],
  'label_axis': -1,
  'test_size': 0.2,
  'limit': 1000,
  'reshape': False,
  'shuffle': True,
  'seed': None},
 'optimizer': {'maximize': False,
  'foreach': None,
  'differentiable': False,
  'weight_decay': 0,
  'name': 'SGD',
  'learning_rate': 0.01,
  'dampening': 0,
  'fused': None,
  'momentum': 0,
  'nesterov': False},
 'loss': {'size_average': None,
  'reduction': 'mean',
  'reduce': None,
  'name': 'MSELoss'},
 'metrics': [],
 'callbacks': {'project': 'unet001',
  'root': '/mnt/storage/projects',
  'monitor': 'val_loss',
  'verbose': 0,
  'save_best_only': False,
  'save_weights_only': False,
  'mode': 'auto',
  'save_freq': 'epoch',
  'initial_value_threshold': None},
 'logger': {'slack_channel': 'dev',
  'slack_url': 'https://hooks.slack.com/services/',
  'slack_methods': ['train'],
  'ti

<font color='#93B6E6'>Any invalid parameters to the entire training process are caught immediately, in a shallow, easy to read traceback.

In [32]:
bad_config = f'''
framework:
    name: torch
    device: cuda
model:
    input_channels: 3
    output_channels: 1
dataset:
    source: /mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001
    ext_regex: exr
    labels: ['a']
    label_axis: -1
    limit: 1000
    test_size: 0.2
    reshape: False
optimizer:
    name: SGD
loss:
    name: MSELoss
metrics:
    - name: KLDivergence
      reduction: fancy            # <-- wrong
callbacks:
    project: unet001
    root: /mnt/storage/projects
train:
    batch_size: 16
logger:
    slack_url: 'https://hooks.slack.com/services/{uri}'
    slack_channel: dev
    slack_methods:
        - train
    timezone: 'America/Detroit'
'''
fi.torch.models.dummy.DummyPipeline.from_string(bad_config).build()

ValidationError: 1 validation error for TorchMetricKLDivergence
reduction
  String should match pattern '^(mean|sum|none)$' [type=string_pattern_mismatch, input_value='fancy', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/string_pattern_mismatch

<font color='#93B6E6'>**generate_config** can be used to generate an initial YAML config given very arguments.

In [49]:
pipe.generate_config(
    framework='tensorflow',
    project='project-name',
    callback_root='/tensorboard/parent/dir',
    dataset='/mnt/data/dataset',
    optimizer='Adam',
    loss='CrossEntropyLoss',
    metrics=['Mean'],
)

callbacks:
  initial_value_threshold: null
  mode: auto
  monitor: val_loss
  project: project-name
  root: /tensorboard/parent/dir
  save_best_only: false
  save_freq: epoch
  save_weights_only: false
  verbose: 0
dataset:
  ext_regex: npy|exr|png|jpeg|jpg|tiff
  label_axis: -1
  labels: null
  limit: null
  reshape: true
  seed: null
  shuffle: true
  source: /mnt/data/dataset
  test_size: 0.2
framework:
  auto_scale_loss: true
  device: cpu
  jit_compile: false
  loss_weights: null
  name: tensorflow
  run_eagerly: false
  steps_per_execution: 1
  weighted_metrics: null
logger:
  level: warn
  slack_channel: null
  slack_methods:
  - load
  - compile
  - train
  slack_url: null
  timezone: UTC
loss:
  axis: -1
  dtype: null
  from_logits: false
  label_smoothing: 0.0
  name: CategoricalCrossentropy
  reduction: sum_over_batch_size
metrics:
- dtype: null
  name: Mean
model: {}
optimizer:
  amsgrad: false
  beta_1: 0.9
  beta_2: 0.99
  clipnorm: null
  clipvalue: null
  ema_momentum: 